# StayNest - Session 6 Assignment (PySpark Deep Dive)
Work through the 8 tasks below in order. Read the Assignment Questions PDF for the
full detail and acceptance criteria. Fill in each `# TODO` cell, run it, and keep the
output visible. Run on Databricks Free Edition (serverless).

## Section 0 - Setup (already done for you)
Upload `bookings.csv`, `hotels.csv`, `customers.csv` to a Volume, then set `BASE`
to that path and run this cell. Counts should be 12000 / 200 / 2000.

In [0]:
# Point BASE at YOUR Volume path
BASE = "/Volumes/cb_catalog/default/staynest_vol"

print(spark.version)

read_csv = lambda name: (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE}/{name}.csv"))

bookings_df   = read_csv("bookings")
hotels_df     = read_csv("hotels")
customers_df  = read_csv("customers")

print(f"bookings: {bookings_df.count()}, "
      f"hotels: {hotels_df.count()}, "
      f"customers: {customers_df.count()}")

4.1.0
bookings: 12000, hotels: 200, customers: 2000


## Task 1 - Read and inspect
Show the schema, a few sample rows, the row count, and summary stats for the
numeric columns of `bookings_df`.

In [0]:
# printSchema, show(5), count, describe(...)
#Display the schema
bookings_df.printSchema()

#prints a sample rows
bookings_df.show(5)

#Prints the row count
bookings_df.count()

#Prints the summary statistics
bookings_df.describe().show()

root
 |-- booking_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- hotel_id: integer (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- nights: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)

+----------+-----------+--------+------------+---------+------+--------+---------+
|booking_id|customer_id|hotel_id|booking_date|     city|nights|  amount|   status|
+----------+-----------+--------+------------+---------+------+--------+---------+
|   9000000|     701600|    3095|  2025-11-27|   Jaipur|     4| 6087.65|completed|
|   9000001|     700065|    3057|  2025-11-06|    Delhi|     1| 8211.19|cancelled|
|   9000002|     701392|    3187|  2025-08-21|   Jaipur|     2| 7176.52|cancelled|
|   9000003|     700867|    3112|  2025-03-22|Bengaluru|     5| 7880.62|completed|
|   9000004|     701521|    3043|  2025-04-19|   Mumbai|     5|21021.51|  pending|
+--------

## Task 2 - Select and filter
From `bookings_df`, select a few useful columns and return the **completed**
bookings with `amount` over 10000 in the cities Goa or Mumbai. Use `col()`, combine
conditions with `&`, and use `.isin(...)`.

In [0]:
# Select and Filter
from pyspark.sql.functions import col

# ── Multiple condition ──
bf=bookings_df.filter(
    (col("amount") > 10000) &
    (col("status") == "completed") &
    (col("city").isin("Goa", "Mumbai"))
).select("booking_id", "customer_id", "hotel_id", "city", "amount", "status")

bf.show(10)

+----------+-----------+--------+------+--------+---------+
|booking_id|customer_id|hotel_id|  city|  amount|   status|
+----------+-----------+--------+------+--------+---------+
|   9000007|     700868|    3127|Mumbai|15693.64|completed|
|   9000010|     700174|    3054|   Goa|30786.36|completed|
|   9000028|     701748|    3022|   Goa|27008.87|completed|
|   9000034|     700110|    3118|   Goa|64388.77|completed|
|   9000050|     701894|    3147|Mumbai|24275.14|completed|
|   9000061|     701454|    3115|Mumbai|19180.93|completed|
|   9000070|     701629|    3118|   Goa|58790.91|completed|
|   9000071|     700768|    3159|Mumbai|47548.35|completed|
|   9000075|     700312|    3006|   Goa|18043.52|completed|
|   9000087|     700953|    3099|   Goa|21529.88|completed|
+----------+-----------+--------+------+--------+---------+
only showing top 10 rows


## Task 3 - Derived columns
Add: `amount_with_gst` (amount plus 12% tax), a `value_tier`
(premium / standard / budget) using `when`/`otherwise`, and a `booking_month`
from `booking_date`.

In [0]:
# Derived Columns
from pyspark.sql.functions import col, when, lit, month, round

# ── Simple derived column (math) ──
bookings_df.withColumn(
    "amount_with_gst",
    round(col("amount") * 1.12, 2)          # 12% GST
).withColumn(
    "value_tier",
    when(col("amount") >= 15000, "premium")
    .when(col("amount") >= 8000, "standard")
    .otherwise("budget")
).withColumn(
    "booking_month",
    month(col("booking_date"))
)

bf=bookings_df.select("booking_id", "amount", "amount_with_gst", "value_tier", "booking_month")
bf.show(10)

+----------+--------+---------------+----------+-------------+
|booking_id|  amount|amount_with_gst|value_tier|booking_month|
+----------+--------+---------------+----------+-------------+
|   9000000| 6087.65|        6818.17|    budget|           11|
|   9000001| 8211.19|        9196.53|  standard|           11|
|   9000002| 7176.52|         8037.7|    budget|            8|
|   9000003| 7880.62|        8826.29|    budget|            3|
|   9000004|21021.51|       23544.09|   premium|            4|
|   9000005|18124.49|       20299.43|   premium|           10|
|   9000006|70999.15|       79519.05|   premium|           11|
|   9000007|15693.64|       17576.88|   premium|            4|
|   9000008| 1492.62|        1671.73|    budget|            1|
|   9000009| 6694.39|        7497.72|    budget|            6|
+----------+--------+---------------+----------+-------------+
only showing top 10 rows


## Task 4 - Aggregations
For **completed** bookings, group by `city` and return: number of bookings, total
revenue, average amount, biggest booking, and the count of unique customers.
Order by revenue, highest first.

In [0]:
# Aggregations
from pyspark.sql.functions import sum, avg, count, max, countDistinct, col, round

agg_df = (bookings_df
    .filter(col("status") == "completed")
    .groupBy("city")
    .agg(
        count(col("booking_id")).alias("num_bookings"),
        round(sum(col("amount")),2).alias("total_revenue"),
        round(avg(col("amount")),2).alias("avg_amount"),
        max(col("amount")).alias("biggest_booking"),
        countDistinct(col("customer_id")).alias("unique_customers")
    )
    .orderBy(col("total_revenue").desc())
)

display(agg_df)

city,num_bookings,total_revenue,avg_amount,biggest_booking,unique_customers
Goa,2546,4.459670179E7,17516.38,78481.24,1441
Mumbai,1715,3.624122112E7,21131.91,78524.52,1153
Delhi,1174,2.631428154E7,22414.21,78669.69,861
Jaipur,979,2.443685313E7,24961.03,76904.87,796
Bengaluru,1318,2.267013697E7,17200.41,78568.81,969
Udaipur,691,1.209442742E7,17502.79,77939.5,592
Rishikesh,407,8606121.58,21145.26,77458.26,363
Manali,480,6235480.68,12990.58,77291.17,409
Munnar,244,3979216.11,16308.26,65177.15,233
Anantapur,106,2257080.65,21293.21,78237.19,105


## Task 5 - Joins
Inner-join bookings to hotels to enrich each booking. Do a left join too. Use
`left_anti` to check for orphaned bookings (expect 0). Then do a three-way join
with customers.

In [0]:
from pyspark.sql.functions import col

# Aliases
b = bookings_df.alias("b")
h = hotels_df.alias("h")
c = customers_df.alias("c")

# ── Inner join ──
enriched = b.join(h, b.hotel_id == h.hotel_id, "inner")
Ij=    enriched.select(
        col("b.booking_id"),
        col("b.hotel_id"),
        col("b.amount"),
        col("h.hotel_name"),
        col("h.city").alias("hotel_city"),
        col("h.category"),
        col("h.star_rating")
    )
Ij.show(5)

# ── Left join ──
left_joined = b.join(h, b.hotel_id == h.hotel_id, "left")
lj=    left_joined.select(
        col("b.booking_id"),
        col("b.hotel_id"),
        col("b.amount"),
        col("h.hotel_name"),
        col("b.city").alias("booking_city"),
        col("h.category"),
        col("h.star_rating")
    )
lj.show(5)

# ── Left anti join ──
orphaned = b.join(h, b.hotel_id == h.hotel_id, "left_anti")
orphaned.show(5)

# ── Three-way join ──
three_way = (
    b.join(h, b.hotel_id == h.hotel_id, "inner")
     .join(c, b.customer_id == c.customer_id, "inner")
)
tw =     three_way.select(
        col("b.booking_id"),
        col("b.hotel_id"),
        col("h.hotel_name"),
        col("c.customer_id"),
        col("c.customer_name"),
        col("b.amount"),
        col("h.category"),
        col("h.star_rating"),
        col("c.membership")
    )

tw.show(5)

+----------+--------+--------+--------------+----------+--------+-----------+
|booking_id|hotel_id|  amount|    hotel_name|hotel_city|category|star_rating|
+----------+--------+--------+--------------+----------+--------+-----------+
|   9000000|    3095| 6087.65| Orchid Suites|    Jaipur|  Budget|        3.8|
|   9000001|    3057| 8211.19|Orchid Retreat|     Delhi|  Luxury|        4.1|
|   9000002|    3187| 7176.52| Marigold Stay|    Jaipur| Premium|        3.8|
|   9000003|    3112| 7880.62|    Grand Stay| Bengaluru|  Budget|        3.6|
|   9000004|    3043|21021.51|  Emerald Stay|    Mumbai| Premium|        4.3|
+----------+--------+--------+--------------+----------+--------+-----------+
only showing top 5 rows
+----------+--------+--------+--------------+------------+--------+-----------+
|booking_id|hotel_id|  amount|    hotel_name|booking_city|category|star_rating|
+----------+--------+--------+--------------+------------+--------+-----------+
|   9000000|    3095| 6087.65| Orc

## Task 6 - Spark SQL + a window function
Register temp views and use `spark.sql` to get revenue by hotel `category` for
completed bookings. Then use a window function to rank the **top 3 hotels by
revenue within each city**.

In [0]:
# Spark SQL and window function

# Register temp views
bookings_df.createOrReplaceTempView("bookings")
hotels_df.createOrReplaceTempView("hotels")

# Revenue by hotel category for completed bookings
display(spark.sql("""
    SELECT h.category, ROUND(SUM(b.amount), 2) AS total_revenue
    FROM bookings b
    JOIN hotels h ON b.hotel_id = h.hotel_id
    WHERE b.status = 'completed'
    GROUP BY h.category
    ORDER BY total_revenue DESC
"""))



category,total_revenue
Luxury,1.068376269E8
Premium,5.825564086E7
Budget,2.233825323E7


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum, rank

window_spec = Window.partitionBy("city").orderBy(col("total_revenue").desc())

# Alias the DataFrames 
b = bookings_df.alias("b") 
h = hotels_df.alias("h")


display(
    b.filter(col("status") == "completed")
    .join(h, "hotel_id")
    .groupBy("b.city", "hotel_name")
    .agg(round(sum("amount"),2).alias("total_revenue"))
    .withColumn("rank", rank().over(window_spec))
    .filter(col("rank") <= 3)
)

city,hotel_name,total_revenue,rank
Anantapur,Azure Retreat,1434328.39,1
Anantapur,Lotus Retreat,617708.27,2
Anantapur,Orchid Residency,205043.99,3
Bengaluru,Serene Inn,1867479.69,1
Bengaluru,Willow Suites,1433492.08,2
Bengaluru,Heritage Stay,1400383.85,3
Delhi,Orchid Retreat,2616612.56,1
Delhi,Marigold Stay,2327675.63,2
Delhi,Orchid Residency,2264974.65,3
Goa,Cedar Residency,5017409.03,1


## Task 7 - Write the result
Write your city-revenue result as **Parquet**, and also as a **Delta table** with
`saveAsTable`. Read the Delta table back to confirm.

In [0]:
# Write city-revenue result as Parquet
ranked_hotels.write.mode("overwrite").parquet(f"{BASE}/output/city_revenue_parquet")
print(f"Written Parquet file to {BASE}/output/city_revenue_parquet")

# Write as Delta table
ranked_hotels.write.mode("overwrite").format("delta").saveAsTable("city_revenue_delta")

# Read Delta table back to confirm
city_revenue_delta_df = spark.table("city_revenue_delta")
display(city_revenue_delta_df)

Written Parquet file to /Volumes/cb_catalog/default/staynest_vol/output/city_revenue_parquet


city,hotel_id,hotel_name,total_revenue,rank
Anantapur,3036,Azure Retreat,1434328.3900000004,1
Anantapur,3189,Lotus Retreat,617708.27,2
Anantapur,3061,Orchid Residency,205043.99,3
Bengaluru,3003,Serene Inn,1633509.0900000005,1
Bengaluru,3162,Willow Suites,1433492.0800000003,2
Bengaluru,3175,Heritage Stay,1400383.85,3
Delhi,3057,Orchid Retreat,2616612.5599999996,1
Delhi,3104,Marigold Stay,2327675.6300000004,2
Delhi,3012,Orchid Residency,2264974.6499999994,3
Goa,3118,Palm Retreat,3112412.420000001,1


## Task 8 - One chained pipeline
In a single chain: keep completed bookings, join hotels, keep hotels with
`star_rating >= 4.0`, group by `city`, sum revenue, order descending, take the
top 5. End with one `.show()`.

In [0]:
# Chained Pipeline

from pyspark.sql.functions import col, sum as F_sum

# Alias the DataFrames 
b = bookings_df.alias("b") 
h = hotels_df.alias("h")

top5_bookings = (b
    .filter(col("status") == "completed")
    .join(h, "hotel_id")
    .filter(col("star_rating") >= 4.0)              # only top-rated restaurants
    .groupBy("b.city")
    .agg(round(sum("amount"),2).alias("revenue"))
    .orderBy(col("revenue").desc())
    .limit(5)
)

top5_bookings.show()


+---------+-------------+
|     city|      revenue|
+---------+-------------+
|      Goa| 2.47253779E7|
|   Mumbai|1.893781767E7|
|    Delhi| 1.85804188E7|
|Bengaluru|   9129192.93|
|  Udaipur|   5916270.08|
+---------+-------------+

